In [10]:
"""
PDF 파싱 파이프라인
- 텍스트 추출 (pdfplumber)
- 이미지 추출 (PyMuPDF)
- 페이지 렌더링 - 벡터 그래픽/구조도 대응 (PyMuPDF)
- 표 추출 (pdfplumber)

결과물: parsed_output/ 폴더에 텍스트, 이미지, 렌더링 페이지 저장
"""

import fitz          # PyMuPDF
import pdfplumber
import json
import os
import base64
from pathlib import Path
from dataclasses import dataclass, field, asdict


In [11]:
# ── 데이터 구조 ────────────────────────────────────────────────────────────────
 
@dataclass
class PageResult:
    page_num:    int    # 1-based
    text:        str    # 추출된 텍스트 (pdfplumber)
    tables:      list   # 마크다운 표 문자열 리스트
    render_path: str    # 페이지 전체 렌더링 경로 → VLM 입력용
    page_type:   str    # "text" / "visual" / "mixed"

In [12]:
# ── 페이지 유형 판단 ───────────────────────────────────────────────────────────
 
def classify_page(fitz_page, text: str) -> str:
    """
    페이지 성격을 분류해서 VLM 처리 여부 결정.
      text   : 텍스트만 있음 → VLM 불필요, 텍스트만 저장
      visual : 이미지/구조도 위주 → VLM 필요
      mixed  : 텍스트 + 시각 요소 혼합 → VLM 필요
    """
    has_text    = len(text.strip()) > 50
    has_images  = len(fitz_page.get_images()) > 0
    has_drawings = len(fitz_page.get_drawings()) > 10  # 벡터 구조도 감지
 
    if has_text and not has_images and not has_drawings:
        return "text"
    if not has_text:
        return "visual"
    return "mixed"
 
 
# ── 텍스트 + 표 추출 ───────────────────────────────────────────────────────────
 
def extract_text_and_tables(pdf_path: str) -> dict[int, dict]:
    """pdfplumber로 텍스트 + 표 추출"""
    results = {}
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            page_num = i + 1
            text     = page.extract_text() or ""
            tables   = page.extract_tables() or []
 
            md_tables = []
            for table in tables:
                if not table:
                    continue
                header = table[0]
                rows   = table[1:]
                md  = "| " + " | ".join(str(c or "") for c in header) + " |\n"
                md += "| " + " | ".join("---" for _ in header) + " |\n"
                for row in rows:
                    md += "| " + " | ".join(str(c or "") for c in row) + " |\n"
                md_tables.append(md)
 
            results[page_num] = {"text": text, "tables": md_tables}
    return results
 
 
# ── 페이지 렌더링 ──────────────────────────────────────────────────────────────
 
def render_page(page: fitz.Page, page_num: int,
                out_dir: Path, dpi: int) -> str:
    """페이지 전체를 PNG로 렌더링 - 벡터 구조도/이미지 모두 포함"""
    mat  = fitz.Matrix(dpi / 72, dpi / 72)
    pix  = page.get_pixmap(matrix=mat)
    path = out_dir / f"p{page_num:03d}.png"
    pix.save(str(path))
    return str(path)
 
 
# ── 메인 파이프라인 ────────────────────────────────────────────────────────────
 
def parse_pdf(pdf_path: str,
              output_dir: str = OUTPUT_DIR,
              dpi: int = PAGE_DPI) -> list[PageResult]:
 
    out     = Path(output_dir)
    ren_dir = out / "pages"
    out.mkdir(parents=True, exist_ok=True)
    ren_dir.mkdir(exist_ok=True)
 
    print(f"📄 PDF 파싱 시작: {pdf_path}")
 
    # 텍스트 + 표 (pdfplumber)
    text_data = extract_text_and_tables(pdf_path)
 
    # 페이지 렌더링 + 분류 (PyMuPDF)
    doc     = fitz.open(pdf_path)
    results = []
 
    print(f"   총 {len(doc)}페이지\n")
    print(f"   {'PAGE':<6} {'유형':<8} {'텍스트':>6}자  {'표':>3}개  렌더링")
    print(f"   {'-'*50}")
 
    for i, page in enumerate(doc):
        page_num = i + 1
        td       = text_data.get(page_num, {"text": "", "tables": []})
        ptype    = classify_page(page, td["text"])
        render   = render_page(page, page_num, ren_dir, dpi)
 
        results.append(PageResult(
            page_num    = page_num,
            text        = td["text"],
            tables      = td["tables"],
            render_path = render,
            page_type   = ptype,
        ))
 
        icon = {"text": "📝", "visual": "🖼️ ", "mixed": "🔀"}[ptype]
        print(f"   p{page_num:03d}   {icon} {ptype:<6}  "
              f"{len(td['text']):>6}자  {len(td['tables']):>3}개  {render}")
 
    doc.close()
 
    # ── 출력 1: 전체 텍스트 (RAG 청킹용) ──────────────────────────────────────
    full_text_path = out / "full_text.txt"
    with full_text_path.open("w", encoding="utf-8") as f:
        for r in results:
            f.write(f"\n{'='*60}\n[PAGE {r.page_num}] ({r.page_type})\n{'='*60}\n\n")
            if r.text:
                f.write(r.text + "\n\n")
            for j, tbl in enumerate(r.tables):
                f.write(f"[표 {j+1}]\n{tbl}\n\n")
 
    # ── 출력 2: VLM 처리 대기열 JSON ──────────────────────────────────────────
    vlm_queue = [
        {
            "page_num":    r.page_num,
            "page_type":   r.page_type,
            "render_path": r.render_path,
            "has_text":    bool(r.text.strip()),
        }
        for r in results
        if r.page_type in ("visual", "mixed")  # text 페이지는 VLM 불필요
    ]
 
    vlm_queue_path = out / "vlm_queue.json"
    vlm_queue_path.write_text(
        json.dumps(vlm_queue, ensure_ascii=False, indent=2)
    )
 
    # ── 출력 3: 전체 메타 요약 ────────────────────────────────────────────────
    summary = {
        "pdf_path":   pdf_path,
        "page_count": len(results),
        "stats": {
            "text_only": sum(1 for r in results if r.page_type == "text"),
            "visual":    sum(1 for r in results if r.page_type == "visual"),
            "mixed":     sum(1 for r in results if r.page_type == "mixed"),
        },
        "vlm_queue_count": len(vlm_queue),
    }
    (out / "summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2)
    )
 
    print(f"\n{'='*50}")
    print(f"✅ 완료: {output_dir}/")
    print(f"   pages/          → 페이지 렌더링 PNG ({len(results)}개)")
    print(f"   full_text.txt   → 텍스트+표 (RAG 청킹용)")
    print(f"   vlm_queue.json  → VLM 처리 대기열 ({len(vlm_queue)}페이지)")
    print(f"   summary.json    → 통계")
    print(f"\n   📊 페이지 분류")
    print(f"      텍스트만  : {summary['stats']['text_only']}페이지 → 텍스트 추출만으로 충분")
    print(f"      시각 위주 : {summary['stats']['visual']}페이지  → VLM 필요")
    print(f"      혼합      : {summary['stats']['mixed']}페이지  → VLM 필요")
 
    return results

In [8]:
# ── 설정 ──────────────────────────────────────────────────────────────────────

PDF_PATH   = "★23년 학교 유무선 운영·관리 안내서_최종.pdf"   # ← 여기에 PDF 경로 입력
OUTPUT_DIR = "parsed_output"
PAGE_DPI   = 200               # 페이지 렌더링 해상도 (구조도는 200~300 추천)


In [14]:
if __name__ == "__main__":
    if not Path(PDF_PATH).exists():
        print(f"❌ PDF 없음: {PDF_PATH}")
    else:
        parse_pdf(PDF_PATH)

📄 PDF 파싱 시작: ★23년 학교 유무선 운영·관리 안내서_최종.pdf
   총 101페이지

   PAGE   유형          텍스트자    표개  렌더링
   --------------------------------------------------
   p001   🖼️  visual      21자    0개  parsed_output\pages\p001.png
   p002   🔀 mixed       77자    1개  parsed_output\pages\p002.png
   p003   🔀 mixed      264자    0개  parsed_output\pages\p003.png
   p004   🔀 mixed      180자    0개  parsed_output\pages\p004.png
   p005   🔀 mixed      210자    0개  parsed_output\pages\p005.png
   p006   🔀 mixed      319자    1개  parsed_output\pages\p006.png
   p007   🔀 mixed      216자    2개  parsed_output\pages\p007.png
   p008   🔀 mixed      217자    1개  parsed_output\pages\p008.png
   p009   🔀 mixed      468자    1개  parsed_output\pages\p009.png
   p010   🔀 mixed      169자    0개  parsed_output\pages\p010.png
   p011   🔀 mixed      231자    0개  parsed_output\pages\p011.png
   p012   🔀 mixed      370자    1개  parsed_output\pages\p012.png
   p013   🔀 mixed      364자    1개  parsed_output\pages\p013.png
   p014   🔀 mixed  